# Pears No. 4

### Importing packages

In [29]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

### Global params

In [30]:
np.random.seed(42)
n_obs = 1_000_000

transition_matrix = np.array([
    [0.95, 0.05],
    [0.15, 0.85]
])

phi_0, sigma_0 = 0.4, 1.0
phi_1, sigma_1 = 0.9, 4.0

### Data

In [31]:
states = np.zeros(n_obs, dtype=int)

for t in range(1, n_obs):
    states[t] = np.random.choice([0, 1], p=transition_matrix[states[t-1]])

spread = np.zeros(n_obs)

for t in range(1, n_obs):
    if states[t] == 0:
        spread[t] = phi_0 * spread[t-1] + np.random.normal(0, sigma_0)
    else:
        spread[t] = phi_1 * spread[t-1] + np.random.normal(0, sigma_1)


### Fit

#### AR(1)

In [34]:
spread_series = pd.Series(spread, name="Spread")
ar_model = sm.tsa.SARIMAX(spread_series, order=(1, 0, 0), trend='c').fit(disp=False)

print(f"AIC: {ar_model.aic:.2f} | BIC: {ar_model.bic:.2f}")
print(f"Estimated Phi (mean reversion): {ar_model.arparams[0]:.4f}")
print(f"Estimated Variance: {ar_model.params['sigma2']:.3f}\n")

print(ar_model.summary())

AIC: 4523265.85 | BIC: 4523301.29
Estimated Phi (mean reversion): 0.7903
Estimated Variance: 5.395

                               SARIMAX Results                                
Dep. Variable:                 Spread   No. Observations:              1000000
Model:               SARIMAX(1, 0, 0)   Log Likelihood            -2261629.924
Date:                Tue, 21 Apr 2026   AIC                        4523265.847
Time:                        16:54:38   BIC                        4523301.294
Sample:                             0   HQIC                       4523275.602
                            - 1000000                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      0.0008      0.002      0.329      0.742      -0.004       0.005
ar.L1          0.7903      0.00

#### MS-AR(1)

In [33]:
ms_model = sm.tsa.MarkovAutoregression(
    spread_series, 
    k_regimes=2,
    order=1,
    switching_variance=True,
    trend='c'
).fit()

print(f"AIC: {ms_model.aic:.2f} | BIC: {ms_model.bic:.2f}")
print("\n")

print("regime 0")
print(f"Mean Reversion (Phi): {ms_model.params['ar.L1[0]']:.4f}")
print(f"Variance (Sigma^2):   {ms_model.params['sigma2[0]']:.4f}")
print("\n")

print("regime 1")
print(f"Mean Reversion (Phi): {ms_model.params['ar.L1[1]']:.4f}")
print(f"Variance (Sigma^2):   {ms_model.params['sigma2[1]']:.4f}")
print("\n")

print("transition matrix")
print(f"P(Stay in Reg 0): {ms_model.params['p[0->0]']:.4f}")
p_1_to_1 = 1 - ms_model.params['p[1->0]']
print(f"P(Stay in Reg 1): {p_1_to_1:.4f}")
print("\n")

print(ms_model.summary())

AIC: 3832705.19 | BIC: 3832799.72


regime 0
Mean Reversion (Phi): 0.4000
Variance (Sigma^2):   0.9996


regime 1
Mean Reversion (Phi): 0.8995
Variance (Sigma^2):   15.9800


transition matrix
P(Stay in Reg 0): 0.9499
P(Stay in Reg 1): 0.8518


                         Markov Switching Model Results                         
Dep. Variable:                   Spread   No. Observations:               999999
Model:             MarkovAutoregression   Log Likelihood            -1916344.596
Date:                  Tue, 21 Apr 2026   AIC                        3832705.192
Time:                          16:53:00   BIC                        3832799.716
Sample:                               0   HQIC                       3832731.204
                               - 999999                                         
Covariance Type:                 approx                                         
                             Regime 0 parameters                              
                 coef    std